# Re-run: a previous attempt and its mark (Paper VI, RQ1)

This notebook recomputes the re-run columns of the RQ1 table from the per-case model outputs published
with release `ai-grader-reliability-2026`, and nothing else: no model call, no corpus, no repository
code. It needs Python 3.9+ with numpy, and `per-case-model-outputs.jsonl` (or the `.gz` the report
offers) in the same folder as this notebook, or in a folder named by `RELEASE_DIR`.

Gemini 3.5 Flash Lite graded the same 150 answers (65 students) under five arms. For each arm:

- **drift**: the mean change from the same grader's mark with no history, on the same answer;
- **bias**: the mean signed error against the teacher (grader minus teacher);
- **MAE**: the mean absolute error against the teacher;
- **QWK**: quadratic weighted kappa with the teacher, both marks rounded half up onto 0..10 (as the
  harness's `metrics.ts` does; Python's `round` is half-to-even, so it is not used).

The run is `grader-vi/history-gemini` in the per-case file. The table's "first run" column comes from
an earlier single run whose per-case output was not kept, so it cannot be recomputed here.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "ai-grader-reliability-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
import numpy as np

ARMS = ["no history (control)", "prior attempt, low grade", "prior attempt, high grade",
        "prior attempt, low, NO mark", "prior attempt, high, NO mark"]
NAMES = {ARMS[1]: "Previous attempt marked low", ARMS[2]: "Previous attempt marked high",
         ARMS[3]: "Low previous attempt, mark removed", ARMS[4]: "High previous attempt, mark removed"}

def load(run):
    rows = [r for r in per_case({run}) if all(r[a] is not None for a in ARMS)]
    h = np.array([r["human"] for r in rows], float)
    X = {a: np.array([r[a] for r in rows], float) for a in ARMS}
    learner = np.array([str(r["learnerId"]) for r in rows])
    return h, X, learner

h, X, learner = load("grader-vi/history-gemini")
print(f"{len(h)} answers graded in all five arms, {len(set(learner))} students")

150 answers graded in all five arms, 65 students


In [3]:
def qwk(h, m, lo=0, hi=10):
    """Quadratic weighted kappa on the integer scale lo..hi, marks rounded half up."""
    k = hi - lo + 1
    a = np.clip(np.floor(h + 0.5) - lo, 0, k - 1).astype(int)
    b = np.clip(np.floor(m + 0.5) - lo, 0, k - 1).astype(int)
    O = np.zeros((k, k))
    np.add.at(O, (a, b), 1)
    E = np.outer(np.bincount(a, minlength=k), np.bincount(b, minlength=k)) / len(a)
    i, j = np.indices((k, k))
    W = (i - j) ** 2 / (k - 1) ** 2
    return 1 - (W * O).sum() / (W * E).sum()

def arm_stats(h, x, c):
    return {"drift": (x - c).mean(), "bias": (x - h).mean(), "mae": np.abs(x - h).mean(), "qwk": qwk(h, x),
            "dQWK": qwk(h, x) - qwk(h, c), "dMAE": np.abs(x - h).mean() - np.abs(c - h).mean()}

ctl = X[ARMS[0]]
point = {a: arm_stats(h, X[a], ctl) for a in ARMS}
print(f"{'Arm':38s} {'drift':>6s} {'bias':>6s} {'MAE':>5s} {'QWK':>6s}")
for a in ARMS:
    p = point[a]
    drift = "    —" if a == ARMS[0] else f"{p['drift']:+.2f}"
    print(f"{NAMES.get(a, 'No history (control)'):38s} {drift:>6s} {p['bias']:+.2f} {p['mae']:5.2f} {p['qwk']:6.3f}")

Arm                                     drift   bias   MAE    QWK
No history (control)                        — +0.35  1.22  0.836
Previous attempt marked low             -0.44 -0.10  1.28  0.825
Previous attempt marked high            -0.16 +0.19  1.20  0.855
Low previous attempt, mark removed      +0.14 +0.49  1.31  0.821
High previous attempt, mark removed     -0.24 +0.11  1.19  0.850


In [4]:
# The paper's printed re-run values (RQ1 table). Point estimates must match exactly.
PRINTED = {  # arm: (drift, bias, MAE, QWK); the control has no drift
    ARMS[0]: (None, 0.35, 1.22, 0.836),
    ARMS[1]: (-0.44, -0.10, 1.28, 0.825),
    ARMS[2]: (-0.16, 0.19, 1.20, 0.855),
    ARMS[3]: (0.14, 0.49, 1.31, 0.821),
    ARMS[4]: (-0.24, 0.11, 1.19, 0.850),
}
for a, (dr, bi, mae, k) in PRINTED.items():
    p = point[a]
    if dr is not None:
        assert round(p["drift"], 2) == dr, (a, "drift", p["drift"], dr)
    assert round(p["bias"], 2) == bi, (a, "bias", p["bias"], bi)
    assert round(p["mae"], 2) == mae, (a, "MAE", p["mae"], mae)
    assert round(p["qwk"], 3) == k, (a, "QWK", p["qwk"], k)

# The text's derived figures.
sd_teacher = h.std()                                   # the draw's teacher SD (population form)
assert round(sd_teacher, 2) == 3.15, sd_teacher
assert round(abs(point[ARMS[1]]["drift"]) / sd_teacher, 2) == 0.14
mark_effect = point[ARMS[3]]["drift"] - point[ARMS[1]]["drift"]   # +0.14 against −0.44
assert round(mark_effect, 1) == 0.6, mark_effect
dq = [point[a]["dQWK"] for a in ARMS[1:]]
dm = [point[a]["dMAE"] for a in ARMS[1:]]
assert (round(min(dq), 3), round(max(dq), 3)) == (-0.015, 0.019), dq
assert (round(min(dm), 2), round(max(dm), 2)) == (-0.03, 0.09), dm
print(f"teacher SD {sd_teacher:.2f}; low-mark drift = {abs(point[ARMS[1]]['drift']) / sd_teacher:.2f} SD; "
      f"mark-removed minus marked (low attempt) = {mark_effect:+.2f}")
print(f"QWK changes {min(dq):+.3f} to {max(dq):+.3f}; MAE changes {min(dm):+.2f} to {max(dm):+.2f}")
print("All 19 printed point values and the derived figures match the paper.")

teacher SD 3.15; low-mark drift = 0.14 SD; mark-removed minus marked (low attempt) = +0.58
QWK changes -0.015 to +0.019; MAE changes -0.03 to +0.09
All 19 printed point values and the derived figures match the paper.


**Intervals.** The study resamples students (the history belongs to the student's answer set):
4,000 resamples of the 65 students with replacement, all of each drawn student's answers, percentile
95% intervals, arm by arm in the table's order from one random generator seeded 11
(`grader-attainment-prior/analyse.py`, `analyse_history`). The cell below uses the same scheme, seed
and order, so on the same numpy version its endpoints are the study's own; the assertions check them
to the printed precision. On a different numpy version the random stream can differ and an endpoint
could move in its last digit, which is why the interval checks report rather than stop.

In [5]:
def cluster_boot(h, X, learner, seed=11, reps=4000):
    rng = np.random.default_rng(seed)
    ids = np.unique(learner)
    members = {g: np.flatnonzero(learner == g) for g in ids}
    out = {}
    for a in ARMS[1:]:
        draws = []
        for _ in range(reps):
            idx = np.concatenate([members[g] for g in rng.choice(ids, size=len(ids), replace=True)])
            draws.append(arm_stats(h[idx], X[a][idx], X[ARMS[0]][idx]))
        out[a] = {k: np.percentile([d[k] for d in draws], [2.5, 97.5]) for k in draws[0]}
    return out

ci = cluster_boot(h, X, learner)
for a in ARMS[1:]:
    c = ci[a]
    print(f"{NAMES[a]:38s} drift {point[a]['drift']:+.2f} [{c['drift'][0]:+.2f}, {c['drift'][1]:+.2f}]"
          f"   dQWK {point[a]['dQWK']:+.3f} [{c['dQWK'][0]:+.3f}, {c['dQWK'][1]:+.3f}]"
          f"   dMAE {point[a]['dMAE']:+.3f} [{c['dMAE'][0]:+.3f}, {c['dMAE'][1]:+.3f}]")

PRINTED_CI = {ARMS[1]: (-0.62, -0.26), ARMS[2]: (-0.33, 0.02), ARMS[3]: (0.02, 0.26), ARMS[4]: (-0.38, -0.10)}
same = all(tuple(np.round(ci[a]["drift"], 2)) == v for a, v in PRINTED_CI.items())
gain_qwk = {a: ci[a]["dQWK"][1] for a in ARMS[1:]}          # largest QWK gain each interval allows
gain_mae = max(-ci[a]["dMAE"][0] for a in ARMS[1:])         # largest MAE improvement allowed
print(f"\nLargest QWK gain the intervals allow: {min(gain_qwk.values()):+.3f} to {max(gain_qwk.values()):+.3f}"
      f" (paper: +0.007 to +0.056); largest MAE improvement {gain_mae:.2f} (paper: 0.17)")
print("Drift intervals equal the printed ones:" if same else
      "Drift intervals differ from the printed ones in the last digit (random stream):", same)

Previous attempt marked low            drift -0.44 [-0.62, -0.26]   dQWK -0.011 [-0.049, +0.024]   dMAE +0.063 [-0.107, +0.240]
Previous attempt marked high           drift -0.16 [-0.33, +0.02]   dQWK +0.019 [-0.012, +0.056]   dMAE -0.017 [-0.169, +0.128]
Low previous attempt, mark removed     drift +0.14 [+0.02, +0.26]   dQWK -0.015 [-0.038, +0.007]   dMAE +0.093 [-0.011, +0.202]
High previous attempt, mark removed    drift -0.24 [-0.38, -0.10]   dQWK +0.014 [-0.013, +0.044]   dMAE -0.030 [-0.163, +0.103]

Largest QWK gain the intervals allow: +0.007 to +0.056 (paper: +0.007 to +0.056); largest MAE improvement 0.17 (paper: 0.17)
Drift intervals equal the printed ones: True


**The independent spot re-run.** The reproduction re-ran the five arms once more (Gemini, the same 150
answers; run `reproduction-rerun/history-gemini`). Read by the same analysis, its unmarked strong
attempt changed MAE by −0.14 [−0.27, −0.02], the one resolved accuracy change the text reports.

In [6]:
h2, X2, learner2 = load("reproduction-rerun/history-gemini")
p2 = arm_stats(h2, X2[ARMS[4]], X2[ARMS[0]])
assert round(p2["dMAE"], 3) == -0.143, p2["dMAE"]
c2 = cluster_boot(h2, X2, learner2)[ARMS[4]]["dMAE"]
print(f"spot re-run, high previous attempt without its mark: dMAE {p2['dMAE']:+.3f} [{c2[0]:+.3f}, {c2[1]:+.3f}]"
      f" (release record: −0.143 [−0.267, −0.023])")

spot re-run, high previous attempt without its mark: dMAE -0.143 [-0.267, -0.023] (release record: −0.143 [−0.267, −0.023])
